# Lab Assignment 1 — Pandas vs. Polars
**ARTI 303 — Programming for AI**

| | |
|---|---|
| **Name** | _[your full name]_ |
| **Student ID** | _[your ID]_ |
| **Section** | _[your section]_ |
| **Date submitted** | _[date]_ |

---

## What this assignment is about

Pandas is not the only way to work with tables in Python, and it is not
always the best one. There is a whole ecosystem out there — **Polars**,
**DuckDB**, **Dask**, **Vaex**, and others — each making different
trade-offs.

The point of this assignment is not to prove one library beats another.
It is to make you **go and find out for yourself**, on data you chose,
and report honestly what you measured — including the parts where the
result surprised you or didn't match what the internet promised.

You will do the same seven operations in both libraries, time them, and
write about what you observed **in your own words**.

---

## What you will submit

**1. This notebook**, completed, run top to bottom, and pushed to your own
GitHub repository. Run it *before* you push, so your results are saved
in the file and can be read without re-running anything.

**2. A Word document** containing, in this order:

1. Your name, student ID, and section
2. A screenshot of you **pulling** this assignment
3. Your dataset link
4. Your GitHub repository link
5. A screenshot of you **pushing** your completed work

### The two screenshots

Both must show your **actual terminal**: the command you typed and the
output it produced. A screenshot of the GitHub website is not accepted.


> Your repository link must open without a login: make the repository public.

---
# Part 1 · Setup

Install Polars into your course environment, with `(.venv)` active in your terminal, then run the cell below.

In [1]:
import pandas as pd
import polars as pl
import numpy as np
import time
import statistics

print("pandas:", pd.__version__)
print("polars:", pl.__version__)

pandas: 3.0.6
polars: 2.0.0


---
# Part 2 · Load your dataset

Pick any public dataset you like, something you are actually curious about. Requirements:

- **At least 100,000 rows.** On small data both libraries finish in under a millisecond and you end up measuring noise, not performance.
- A **CSV file**
- At least **one text/category column** (to group and join on)
- At least **two numeric columns** (to compute with)

Kaggle, data.gov.sa, the UCI Machine Learning Repository, and Hugging
Face Datasets are all good places to look. If the file is over about
50 MB, do not commit it: add it to `.gitignore` and put the download
link in your Word document.

Set `DATA_PATH` to your file, then run the cell.

In [2]:
DATA_PATH = "flights.csv"

df_pandas = pd.read_csv(DATA_PATH)
df_polars = pl.read_csv(DATA_PATH)

print("pandas shape:", df_pandas.shape)
print("polars shape:", df_polars.shape)
print()
print("columns:", list(df_pandas.columns))
print()
print(df_pandas.head(3))

rows = len(df_pandas)
print()
print(f"rows: {rows:,}")
if rows < 100_000:
    print("TOO SMALL - timing results on this dataset will be mostly noise.")
    print("Pick a larger dataset before continuing.")
else:
    print("Size is fine.")

C:\Users\HP\AppData\Local\Temp\ipykernel_16396\2317235107.py:3: DtypeWarning: Columns (7: ORIGIN_AIRPORT, 8: DESTINATION_AIRPORT) have mixed types. Specify dtype option on import or set low_memory=False.
  df_pandas = pd.read_csv(DATA_PATH)


pandas shape: (5819079, 31)
polars shape: (5819079, 31)

columns: ['YEAR', 'MONTH', 'DAY', 'DAY_OF_WEEK', 'AIRLINE', 'FLIGHT_NUMBER', 'TAIL_NUMBER', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'SCHEDULED_DEPARTURE', 'DEPARTURE_TIME', 'DEPARTURE_DELAY', 'TAXI_OUT', 'WHEELS_OFF', 'SCHEDULED_TIME', 'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'WHEELS_ON', 'TAXI_IN', 'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME', 'ARRIVAL_DELAY', 'DIVERTED', 'CANCELLED', 'CANCELLATION_REASON', 'AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY', 'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY']

   YEAR  MONTH  DAY  DAY_OF_WEEK AIRLINE  FLIGHT_NUMBER TAIL_NUMBER  \
0  2015      1    1            4      AS             98      N407AS   
1  2015      1    1            4      AA           2336      N3KUAA   
2  2015      1    1            4      US            840      N171US   

  ORIGIN_AIRPORT DESTINATION_AIRPORT  SCHEDULED_DEPARTURE  ...  ARRIVAL_TIME  \
0            ANC                 SEA                    5  ...         408.0   

---
# Part 3 · How to time things properly


### Rule 1 — Run it more than once, take the median

A single run is affected by whatever else your computer was doing at that moment. Run each operation several times and take the middle value.

### Rule 2 — Polars is lazy. You must call `.collect()`

Polars has two modes. In **lazy** mode (`pl.scan_csv`, `.lazy()`), nothing actually runs when you write the query — Polars just records your plan. The work only happens when you call `.collect()`.

So if you time a lazy query without `.collect()`, you are timing the act of *writing down a plan*, not executing it. You will get a number close to zero and conclude Polars is hundreds of times faster. It is not. You measured nothing.

### Rule 3 — The first read of a file is slower

Your operating system caches file contents after the first read. Read the file once before you start timing, so both libraries are measured under the same conditions.

### The timing helper

Use this function for every measurement below.

In [3]:
def measure(operation, repeats=5):
    """Run `operation` several times and return the median time in milliseconds.

    operation : a function taking no arguments
    """
    # warm-up run, not measured
    operation()

    timings = []
    for _ in range(repeats):
        start = time.perf_counter()
        operation()
        timings.append(time.perf_counter() - start)

    return statistics.median(timings) * 1000

### See Rule 2 for yourself

Run this and look at the difference.

In [4]:
lazy_no_collect = measure(lambda: pl.scan_csv(DATA_PATH).filter(pl.col(df_pandas.columns[0]).is_not_null()))
lazy_collected  = measure(lambda: pl.scan_csv(DATA_PATH).filter(pl.col(df_pandas.columns[0]).is_not_null()).collect())

print(f"lazy WITHOUT .collect(): {lazy_no_collect:8.3f} ms   <- nothing was executed")
print(f"lazy WITH .collect():    {lazy_collected:8.3f} ms   <- the real cost")

lazy WITHOUT .collect():    0.020 ms   <- nothing was executed
lazy WITH .collect():     683.281 ms   <- the real cost


**Record what you saw:**

| | ms |
|---|---|
| Lazy query without `.collect()` | _[0.040 ms]_ |
| Lazy query with `.collect()` | _[975.922 ms] _ |

In one sentence — why is the first number meaningless?

_[Because it only measures the time taken to build the query plan without actually reading or processing any data.]_

In [5]:
# Set these to real column names from YOUR dataset
CATEGORY_COL = "ORIGIN_AIRPORT"   # text/category column for grouping
NUMERIC_COL  = "DEPARTURE_DELAY" # numeric column
NUMERIC_COL2 = "ARRIVAL_DELAY"   # second numeric column

for col in [CATEGORY_COL, NUMERIC_COL, NUMERIC_COL2]:
    if col not in df_pandas.columns:
        print(f"[!] '{col}' is not in your dataset - update it above")
    else:
        print(f"[ok] {col}")

[ok] ORIGIN_AIRPORT
[ok] DEPARTURE_DELAY
[ok] ARRIVAL_DELAY


## 4.1 — Reading the file

Time reading the whole file from disk in each library.

In [6]:
pandas_time = measure(lambda: pd.read_csv(DATA_PATH))
polars_time = measure(lambda: pl.read_csv(DATA_PATH))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

read_pandas, read_polars = pandas_time, polars_time

C:\Users\HP\AppData\Local\Temp\ipykernel_16396\38132050.py:1: DtypeWarning: Columns (7: ORIGIN_AIRPORT, 8: DESTINATION_AIRPORT) have mixed types. Specify dtype option on import or set low_memory=False.
  pandas_time = measure(lambda: pd.read_csv(DATA_PATH))


pandas:  9418.63 ms
polars:   491.10 ms
ratio:     19.18x


**Observation:** _[Polars loaded the CSV file significantly faster than Pandas due to its parallelized Rust-based arrow engine. Pandas was slower as it scans the whole file sequentially to infer column data types.]_

## 4.2 — Filtering rows

Keep only rows where your numeric column is above its median.

In [7]:
threshold = df_pandas[NUMERIC_COL].median()
print("threshold:", threshold)

pandas_time = measure(lambda: df_pandas[df_pandas[NUMERIC_COL] > threshold])
polars_time = measure(lambda: df_polars.filter(pl.col(NUMERIC_COL) > threshold))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

filter_pandas, filter_polars = pandas_time, polars_time

threshold: -2.0
pandas:   619.68 ms
polars:   127.62 ms
ratio:      4.86x


**Observation:** _[Polars is faster at filtering rows because it uses a column-based memory layout and multiple CPU threads, whereas Pandas relies mainly on single-threaded filtering.]_

## 4.3 — Creating a new column

Compute a new column from two existing ones.

In [8]:
pandas_time = measure(
    lambda: df_pandas.assign(new_col=df_pandas[NUMERIC_COL] - df_pandas[NUMERIC_COL2])
)
polars_time = measure(
    lambda: df_polars.with_columns(
        (pl.col(NUMERIC_COL) - pl.col(NUMERIC_COL2)).alias("new_col")
    )
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

newcol_pandas, newcol_polars = pandas_time, polars_time

pandas:    17.32 ms
polars:    25.32 ms
ratio:      0.68x


**Observation:** _[Pandas was slightly faster for this simple calculation because it has less setup overhead, whereas Polars takes a little extra time to set up its parallel processing engine]_

## 4.4 — Group by and aggregate

Group by your category column and compute a sum and a mean.

In [9]:
pandas_time = measure(
    lambda: df_pandas.groupby(CATEGORY_COL).agg(
        total=(NUMERIC_COL, "sum"),
        average=(NUMERIC_COL, "mean"),
    )
)
polars_time = measure(
    lambda: df_polars.group_by(CATEGORY_COL).agg([
        pl.col(NUMERIC_COL).sum().alias("total"),
        pl.col(NUMERIC_COL).mean().alias("average"),
    ])
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

groupby_pandas, groupby_polars = pandas_time, polars_time

# Confirm both libraries agree on the answer
print()
print("pandas result (first rows):")
print(df_pandas.groupby(CATEGORY_COL)[NUMERIC_COL].sum().head())
print()
print("polars result (first rows):")
print(df_polars.group_by(CATEGORY_COL).agg(pl.col(NUMERIC_COL).sum()).head())

pandas:   514.71 ms
polars:   194.69 ms
ratio:      2.64x

pandas result (first rows):
ORIGIN_AIRPORT
10135    2277.0
10136     613.0
10140    9700.0
10141     119.0
10146     779.0
Name: DEPARTURE_DELAY, dtype: float64

polars result (first rows):
shape: (5, 2)
┌────────────────┬─────────────────┐
│ ORIGIN_AIRPORT ┆ DEPARTURE_DELAY │
│ ---            ┆ ---             │
│ str            ┆ i64             │
╞════════════════╪═════════════════╡
│ SDF            ┆ 88445           │
│ 14256          ┆ 399             │
│ 14986          ┆ 2399            │
│ 13244          ┆ 4201            │
│ 10423          ┆ 30547           │
└────────────────┴─────────────────┘


**Observation:** _[pandas was faster in group by (259.73ms vs 384.95ms, ratio is 0.67x). polars was not faster here because the category column ORIGIN_AIRPORT is text and has many unique values, so the grouping on it costs more.]_

**Do both libraries give the same numbers?** _[not in the same order. pandas sorts the groups by the key by default (10135,10136,....,10146), while polars returns them without fixes order, so the first rows i printed are different. The values are the same once sorted, but i should sort both before comaring.]_

## 4.5 — Sorting and taking the top N

In [10]:
pandas_time = measure(
    lambda: df_pandas.sort_values(NUMERIC_COL, ascending=False).head(10)
)
polars_time = measure(
    lambda: df_polars.sort(NUMERIC_COL, descending=True).head(10)
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

sort_pandas, sort_polars = pandas_time, polars_time

pandas:  3237.46 ms
polars:   357.57 ms
ratio:      9.05x


**Observation:** _[polars was faster here at sorting and it takes the first top 10 rows(1512.64ms vs 2355.13ms for pandas, ratio of 1.56x). The difference is real but not very large. Sorting has to look at every row of the dataset, so both libraries nedd a lot of work, and i do think that polars is faster because it sorts using multiple CPU cores.]_

## 4.6 — The Lazy API

This one has no Pandas equivalent, which is the point. Pandas executes every step immediately. Polars can build the whole plan first, optimize it, and only then run it — reading just the columns it needs and pushing
filters as early as possible.

Chain several operations together lazily, then `.collect()`.

In [11]:
def polars_lazy_pipeline():
    return (
        pl.scan_csv(DATA_PATH)
          .filter(pl.col(NUMERIC_COL) > threshold)
          .with_columns((pl.col(NUMERIC_COL) - pl.col(NUMERIC_COL2)).alias("new_col"))
          .group_by(CATEGORY_COL)
          .agg(pl.col("new_col").sum().alias("total"))
          .sort("total", descending=True)
          .collect()
    )

def pandas_equivalent():
    d = pd.read_csv(DATA_PATH)
    d = d[d[NUMERIC_COL] > threshold]
    d = d.assign(new_col=d[NUMERIC_COL] - d[NUMERIC_COL2])
    d = d.groupby(CATEGORY_COL).agg(total=("new_col", "sum"))
    return d.sort_values("total", ascending=False)

pandas_time = measure(pandas_equivalent, repeats=3)
polars_time = measure(polars_lazy_pipeline, repeats=3)

print(f"pandas (step by step): {pandas_time:8.2f} ms")
print(f"polars (lazy + collect): {polars_time:8.2f} ms")
print(f"ratio: {pandas_time/polars_time:8.2f}x")

lazy_pandas, lazy_polars = pandas_time, polars_time

C:\Users\HP\AppData\Local\Temp\ipykernel_16396\2977073395.py:13: DtypeWarning: Columns (7: ORIGIN_AIRPORT, 8: DESTINATION_AIRPORT) have mixed types. Specify dtype option on import or set low_memory=False.
  d = pd.read_csv(DATA_PATH)


pandas (step by step): 11076.22 ms
polars (lazy + collect):   366.90 ms
ratio:    30.19x


**Look at the query plan.** Polars can show you what it decided to do.

In [12]:
plan = (
    pl.scan_csv(DATA_PATH)
      .filter(pl.col(NUMERIC_COL) > threshold)
      .group_by(CATEGORY_COL)
      .agg(pl.col(NUMERIC_COL).sum())
)

print(plan.explain())

AGGREGATE[maintain_order: false]
  [col("DEPARTURE_DELAY").sum()] BY [col("ORIGIN_AIRPORT")]
  FROM
  Csv SCAN [flights.csv]
  PROJECT 2/31 COLUMNS
  SELECTION: col("DEPARTURE_DELAY").cast(Float64) > -2.0
  ESTIMATED ROWS: 5807908


**Observation:** _[polars was much faster (544.17ms vs 8600.71ms, and abouth 15.81x of ratio). The plan shows it reads only 2 of 31 columns and applies the filter inside the CSV scan, while pandas loads the wholefile first and then runs each step.]_

## 4.7 — Joining two tables

Build a small lookup table from your data, then join it back.

In [13]:
# a lookup table: average value per category
lookup_pandas = df_pandas.groupby(CATEGORY_COL, as_index=False)[NUMERIC_COL].mean()
lookup_pandas = lookup_pandas.rename(columns={NUMERIC_COL: "category_average"})

lookup_polars = (
    df_polars.group_by(CATEGORY_COL)
             .agg(pl.col(NUMERIC_COL).mean().alias("category_average"))
)

pandas_time = measure(lambda: df_pandas.merge(lookup_pandas, on=CATEGORY_COL, how="left"))
polars_time = measure(lambda: df_polars.join(lookup_polars, on=CATEGORY_COL, how="left"))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

join_pandas, join_polars = pandas_time, polars_time

# Row-count check - a join should not change the number of rows here
merged = df_pandas.merge(lookup_pandas, on=CATEGORY_COL, how="left")
print()
print("rows before:", len(df_pandas), "-> after:", len(merged))
print("row count unchanged:", len(merged) == len(df_pandas))

pandas:   611.87 ms
polars:    31.52 ms
ratio:     19.41x

rows before: 5819079 -> after: 5819079
row count unchanged: True


**Observation:** _[polars was much faster at the join (39.46ms vs 968.56ms for pandas, and about 24.55x of ratio). both kept the same number of rows (5819079), so the joined worked correctly. This was the biggest gap in the operation i ran.]_

---
# Part 5 · Results summary

Collect every measurement into one table.

In [14]:
results = pd.DataFrame({
    "operation": ["read file", "filter", "new column", "group by",
                  "sort + top N", "chained pipeline", "join"],
    "pandas_ms": [read_pandas, filter_pandas, newcol_pandas, groupby_pandas,
                  sort_pandas, lazy_pandas, join_pandas],
    "polars_ms": [read_polars, filter_polars, newcol_polars, groupby_polars,
                  sort_polars, lazy_polars, join_polars],
})

results["speedup"] = (results["pandas_ms"] / results["polars_ms"]).round(2)
results["faster"] = np.where(results["speedup"] > 1, "polars", "pandas")

results.round(2)

,operation,pandas_ms,polars_ms,speedup,faster
0,read file,9418.63,491.10,19.18,polars
1,filter,619.68,127.62,4.86,polars
2,new column,17.32,25.32,0.68,pandas
3,group by,514.71,194.69,2.64,polars
4,sort + top N,3237.46,357.57,9.05,polars
5,chained pipeline,11076.22,366.90,30.19,polars
6,join,611.87,31.52,19.41,polars


In [15]:
# Save your results so they are in the repository as evidence
import os
os.makedirs("results", exist_ok=True)
results.round(3).to_csv("results/benchmark.csv", index=False)
print("saved to results/benchmark.csv")

print()
print("Machine this was measured on:")
import platform
print("  ", platform.platform())
print("  ", platform.processor() or "processor not reported")

saved to results/benchmark.csv

Machine this was measured on:
   Windows-11-10.0.26200-SP0
   ARMv8 (64-bit) Family 8 Model 1 Revision 201, Qualcomm Technologies Inc


> Recording the machine matters. Your classmate's numbers will differ
> from yours, and neither of you is wrong — timing results are only
> meaningful alongside the conditions they were measured under.

---
# Part 6 · Your analysis

Write in **your  words**. Short, specific answers are better than long vague ones. Answers that could have been written without running anything will not receive credit.

### Q1 — Where was the difference largest, and where was it smallest?

The largest difference was in the chained pipeline, where Polars was 30.19 times faster than Pandas. I think this is because Polars can optimize multiple operations together and read only the columns it needs.

The smallest difference was in creating a new column, where Pandas was slightly faster, taking 17.32 ms compared to 25.32 ms for Polars.

### Q2 — Did anything surprise you?

I was surprised that Polars was much faster in the chained pipeline I also expected Polars to be faster in every operation, but Pandas was faster when creating a new column.

This showed me that the performance depends on the type of operation, not just the library.

### Q3 — What was harder to write?

I found Pandas easier to understand at first, especially for filtering rows and creating new columns. Polars syntax was a little different, so I had to learn methods like `filter()`, `with_columns()`, and `group_by()`.

### Q4 — The marketing claim

Yes, I observed speedups greater than 5x in three operations: reading the file (19.18x), sorting and taking the top 10 rows (9.05x), and the chained pipeline (30.19x). However, Polars was only 4.86x faster for filtering, and Pandas was faster when creating a new column. 

The results may differ from published benchmarks because performance depends on the dataset, hardware, library versions

### Q5 — Your recommendation

I would choose Pandas for small or medium-sized datasets and projects that need straightforward data analysis, because its syntax is easy to use and it has a wide range of tools. I would choose Polars for large datasets or workflows with several operations, especially when performance is important.

### Q6 — What else is out there?

Library: DuckDB

Documentation: https://duckdb.org/docs/

DuckDB is an analytical database designed to run SQL queries efficiently on large datasets. It can query data files such as CSV and Parquet directly, making it useful for data analysis without needing to load the entire dataset into a traditional database first.

---
# Submission checklist

- [ ] Part 1: `polars` installed and recorded in `requirements.txt`
- [ ] Part 2: dataset loaded, with 100,000+ rows
- [ ] Part 3: lazy `.collect()` comparison run and the question answered
- [ ] Part 4: all seven operations run in both libraries, each with an observation
- [ ] Part 5: results table produced and saved to `results/benchmark.csv`
- [ ] Part 6: all six questions answered in your own words
- [ ] **Restart Kernel and Run All**: no errors, and the outputs are visible
- [ ] Notebook pushed to your GitHub repository
- [ ] Word document submitted with: name/ID/section, pull screenshot, dataset link, GitHub link, push screenshot